# SQLDiagnose — train CodeBERT on a GPU (Colab / Kaggle)

The local model (`CodeBERTa-small`) is trained on CPU. This notebook trains the larger
`microsoft/codebert-base` on a free GPU (~20–30 min on a T4) and evaluates it with the same
code and the same data splits, so the numbers are directly comparable.

**Runtime → Change runtime type → GPU** before running.

## 1. Get the project
Either clone your repository, or upload a zip of the project folder (without `.venv/` and `data/raw/`).

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<you>/SQLDiagnose.git" - leave empty to upload a zip instead
import os
if REPO_URL:
    !git clone -q $REPO_URL SQLDiagnose
else:
    from google.colab import files          # on Kaggle: add the zip as a dataset and unzip it instead
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !unzip -q -o "$zip_name" -d SQLDiagnose_upload
    inner = [d for d in os.listdir("SQLDiagnose_upload") if os.path.isdir(os.path.join("SQLDiagnose_upload", d))]
    src = os.path.join("SQLDiagnose_upload", inner[0]) if len(inner) == 1 and not os.path.exists("SQLDiagnose_upload/analysis") else "SQLDiagnose_upload"
    !mv "$src" SQLDiagnose
%cd SQLDiagnose

In [ ]:
!pip install -q "sqlglot>=25" sentencepiece
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Dataset
The processed splits ship with the project, so the GPU model is trained and tested on exactly the same data as the local one. They are only rebuilt (from Spider, ~200 MB) if missing.

In [ ]:
import os
if os.path.exists("data/processed/train.jsonl"):
    print("Using the shipped dataset (identical splits to the local run).")
else:
    !python -m data_pipeline.build_dataset --download

## 3. Fine-tune CodeBERT

In [ ]:
!python -m training.train --model microsoft/codebert-base \
    --output-dir models/checkpoints/codebert-base \
    --epochs 4 --batch-size 32 --lr 3e-5 --eval-steps 100 --patience 5 --threads 0

## 4. Evaluate (same protocol as the local report)
The analyzer audit and NL2SQL sections need the raw Spider databases; run the download once if you want them.

In [ ]:
!python -m data_pipeline.build_dataset --download-only
!python -m evaluation.evaluate --model-dir models/checkpoints/codebert-base/best --nl2sql-samples 150 --out reports_gpu

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("reports_gpu/evaluation_report.md").read()))

## 5. Download the checkpoint and report
Unzip into the project root locally, then either set `api.model_dir` in `configs/api_config.yaml` to `models/checkpoints/codebert-base/best`, or compare both models with
`python -m evaluation.evaluate --model-dir models/checkpoints/codebert-base/best --model-dir models/checkpoints/codeberta-small/best`.

In [ ]:
!zip -q -r codebert-base.zip models/checkpoints/codebert-base/best models/checkpoints/codebert-base/train_log.csv \
    models/checkpoints/codebert-base/training_summary.json reports_gpu
try:
    from google.colab import files
    files.download("codebert-base.zip")
except ImportError:
    print("Download codebert-base.zip from the output panel.")